# Adam's Theory A-Share Stock Picker

Based on J. Welles Wilder Jr.'s **Adam's Theory of Markets**.

**Signal rule**: >= 2 of 3 conditions must be met for a buy signal.

| # | Condition | Detection |
|---|-----------|-----------|
| 1 | Breakout | Close >= highest high of last 20 bars |
| 2 | Trend Change | Prior downtrend/consolidation -> price breaks above swing high |
| 3 | Gap / Wide Range | Gap up >= 0.5% OR range >= 1.5x 20-day avg |

**Stop loss**: lowest low in last 40 bars (structural support).

In [1]:
import sys, io
sys.stdout = io.TextIOWrapper(sys.stdout.buffer, encoding='utf-8')
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

AttributeError: 'OutStream' object has no attribute 'buffer'

## 1. Load Data

In [ ]:
from data.consolidated_store import load_all_stocks, filter_active_stocks
from config.settings import settings

df = load_all_stocks()
print(f"Loaded {df.symbol.nunique():,} stocks, {len(df):,} rows")
print(f"Date range: {df.date.min().date()} ~ {df.date.max().date()}")
print(f"Columns: {list(df.columns)}")
df.head()

## 2. Center Symmetry Projection

In [ ]:
from indicators.adams_theory import compute_center_symmetry_projection

# Example: China Merchants Bank
stock = df[df['symbol'] == '600036'].sort_values('date')
proj = compute_center_symmetry_projection(stock, lookback=20)

print(f"Stock: {stock.iloc[0]['name']} (600036)")
print(f"Anchor price: {proj.anchor_price:.2f}")
print(f"Direction: {proj.projected_direction.upper()}")
print(f"Convergence: {proj.convergence_score:.3f} (1.0 = tight cluster)")
print(f"Projection range: {min(proj.projected_prices):.2f} ~ {max(proj.projected_prices):.2f}")
print(f"First projected bar: {proj.projected_prices[0]:.2f}")

## 3. Three Condition Detectors

In [ ]:
from indicators.adams_theory import (
    detect_breakout, detect_trend_change, detect_gap_or_wide_range
)

symbol = '600036'
stock = df[df['symbol'] == symbol].sort_values('date')
print(f"{stock.iloc[0]['name']} ({symbol}) — latest close: {stock.iloc[-1]['close']:.2f}\n")

c1 = detect_breakout(stock)
c2 = detect_trend_change(stock)
c3 = detect_gap_or_wide_range(stock)

for label, clue in [('Breakout', c1), ('Trend Change', c2), ('Gap/Wide Range', c3)]:
    if clue:
        print(f"  [OK] {label}: {clue.detail}")
    else:
        print(f"  [--] {label}: not triggered")

met = sum(1 for x in [c1, c2, c3] if x)
print(f"\n  {met}/3 conditions met -> {'BUY SIGNAL' if met >= 2 else 'NO SIGNAL'}")

## 4. Signal Detection Pipeline

In [ ]:
from signals.detector import detect_signal
from signals.scorer import compute_quality_score

stock = df[df['symbol'] == '600036'].sort_values('date')
signal = detect_signal(stock, symbol='600036', name=stock.iloc[0]['name'])

if signal:
    quality = compute_quality_score(signal)
    stop_pct = (signal.current_close - signal.stop_loss_price) / signal.current_close * 100
    print(f"BUY: {signal.name} ({signal.symbol})")
    print(f"  Close: {signal.current_close:.2f} | Stop: {signal.stop_loss_price:.2f} (-{stop_pct:.1f}%)")
    print(f"  Risk: {signal.risk_score}/10 | Quality: {quality:.0f}/100")
    print(f"  Projection: {signal.projection.projected_direction.upper()} | Conv: {signal.projection.convergence_score:.3f}")
    print(f"  Volume ratio: {signal.volume_ratio:.1f}x")
    for clue in signal.clues:
        print(f"  [{clue.strength:.2f}] {clue.clue_type}")
else:
    print("No buy signal for 600036")

## 5. Scan All Stocks & Generate Recommendations

In [ ]:
from signals.detector import detect_signal
from recommendation.ranking import select_top_recommendations
from recommendation.explainer import build_explanation, brief_reason
import time

# Filter to most active stocks
candidates = filter_active_stocks(df, top_n=200)
print(f"Scanning {len(candidates)} most active stocks...\n")

t0 = time.time()
signals = []
for i, c in enumerate(candidates):
    sym = c['symbol']
    name = c.get('name', '')
    stock = df[df['symbol'] == sym].sort_values('date')
    if len(stock) < 60:
        continue
    sig = detect_signal(stock, symbol=sym, name=name)
    if sig:
        signals.append(sig)

elapsed = time.time() - t0
print(f"Scanned {len(candidates)} stocks in {elapsed:.1f}s")
print(f"Signals found: {len(signals)}")

# Rank and select top 20
recs = select_top_recommendations(signals, top_n=20)
for r in recs:
    r.reason = build_explanation(r)

# Build display table
rows = []
for i, r in enumerate(recs, 1):
    clue_map = {'breakout': 'B', 'trend_change': 'T', 'range_expansion': 'R'}
    met = '+'.join(clue_map[c.clue_type] for c in r.clues)
    all_types = {'breakout', 'trend_change', 'range_expansion'}
    met_types = {c.clue_type for c in r.clues}
    missed = ','.join(clue_map[t] for t in all_types - met_types)
    stop_pct = (r.current_close - r.stop_loss_price) / r.current_close * 100

    rows.append({
        '#': i,
        'Code': r.symbol,
        'Name': r.name,
        'Close': f"{r.current_close:.2f}",
        'Met': met,
        'Miss': missed,
        'Proj': r.projection.projected_direction[:2].upper(),
        'Risk': f"{r.risk_score:.1f}",
        'Stop%': f"-{stop_pct:.1f}%",
    })

result_df = pd.DataFrame(rows)
print(f"\nTop {len(recs)} Recommendations:")
result_df

## 6. Top Pick — Full Reasoning

In [ ]:
if recs:
    print(recs[0].reason)